# 📘Домашнє завдання №11 Логістична регресія

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW11

Використайте датасет: **Breast Cancer (sklearn)**.

Ціль: передбачити, чи є пухлина злоякісною:
* 1 — злоякісна (malignant)
* 0 — доброякісна (benign)

Завдання:
1. Розділіть на train/test (80/20) і масштабуйте ознаки (StandardScaler).
2. Натренуйте логістичну регресію, порахуйте для тестового датасету:
    * Confusion Matrix
    * Accuracy
    * Precision
    * Recall
    * F1-score
    * ROC-AUC
3. Побудуйте:
    * ROC-криву
    * Precision-Recall криву
4. Знайдіть коефіціенти моделі і шанси (odds).
5. Змініть threshold, для кожного порахуйте precision/recall:
    * 0.3
    * 0.5
    * 0.7

In [1]:
# Download data (to cover the case when the data aren't accessible)

# !pip install --upgrade nbformat
# !pip install jinja2

import shutil
import sqlite3
import pandas as pd
from sklearn.datasets import load_breast_cancer
from pathlib import Path

# Input data
ds_name = "breast_cancer"
db_file_name = "store_hw11.db"
git_project_url = "https://github.com/BogdanPinchuk/DataScience-PBY_HW11.git"
main_file_name = "Bohdan_Pinchuk_DS_HW11.ipynb"

# Solution

# Note: to handle error: "SSL: CERTIFICATE_VERIFY_FAILED" or no connection to the server
try:
    # for testing
    # raise Exception
    ds_sklearn = load_breast_cancer()

    feature_col_names = [name.replace(' ', '_') for name in ds_sklearn["feature_names"]]
    targets = [ds_sklearn["target_names"][idx] for idx in ds_sklearn["target"]]
    ds_data = pd.DataFrame(ds_sklearn["data"], columns=feature_col_names)
    ds_data["target"] = targets

    # # Use only one time to initialize/update data (at first time)
    # conn = sqlite3.connect(db_file_name)
    # ds_data.to_sql(ds_name, conn, if_exists="replace", index=False)
    # conn.close()
except Exception:
    file_path = Path(db_file_name)

    if not file_path.exists():
        # upload all files
        current_path = !pwd
        current_path = current_path[0]
        parent_path = !dirname "$current_path"
        parent_path = parent_path[0]
        temp_path = f"{parent_path}/temp"

        # Clone data
        !rm -rf "$temp_path"
        !git clone "$git_project_url" "$temp_path"

        source = Path(temp_path)
        destination = Path(current_path)
        exclude = {main_file_name, ".git", ".idea"}

        for item in source.iterdir():
            if item.name in exclude:
                continue

            target = destination / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)

        # Clean temp folder
        !rm -rf "$temp_path"

    conn = sqlite3.connect(db_file_name)
    ds_data = pd.read_sql(f"SELECT * FROM {ds_name}", conn)
    conn.close()

display(ds_data)

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,malignant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,malignant
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,malignant
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,malignant
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,malignant


In [2]:
# display(ds_sklearn)
# display(ds_data["data"])
# display(ds_data["target"])
# display(ds_data["target_names"])
# feature_names = [name.replace(' ', '_') for name in ds_data["feature_names"]]
# print(feature_names)

# import pandas as pd
# from sklearn.datasets import load_breast_cancer
#
# breast_cancer = load_breast_cancer()
#
# df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
# df['target'] = breast_cancer.target
#
# df.head(50)